<a href="https://colab.research.google.com/github/vermasachin6102/JoyAI-Echo/blob/main/seed_veo_3__joy_ai_echo_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

# NOTE: deliberately no `import torch` here. Importing torch this early
# caches the base-image's preinstalled version (e.g. 2.11.0) in this
# kernel's memory -- a later `pip install --force-reinstall torch==2.8.0`
# in the setup cell writes the right version to disk, but this already-
# running process would keep returning the cached module on any later
# `import torch`, silently ignoring the pin. Parsing nvidia-smi directly
# avoids importing torch before the pinned install runs.
import subprocess

_result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
    capture_output=True, text=True, check=True,
)
_name, _mem_mib = [x.strip() for x in _result.stdout.strip().split(",")]
vram_gb = float(_mem_mib) / 1024
print(f"GPU: {_name} | VRAM: {vram_gb:.1f} GB")
if vram_gb < 22:
    print("WARNING: <22GB VRAM -- too small for either stage. Switch to L4/A100 (Runtime > Change runtime type).")
elif vram_gb < 40:
    print("24GB-class GPU (e.g. L4): both stages have a quantized path, both ON by default below.")
    print("  Stage 1 (text encoder): Q4_0 GGUF -> NF4. Measured ~13GB peak. Confirmed working.")
    print("  Stage 2 (generator):    FP8 downcast of ~36.5GB bf16. Fits only if fp8 holds.")
    print("     Watch for this line in the run log:")
    print("       [Stage 2] generator param element counts by dtype: {...}")
    print("     float8_e4m3fn present -> fp8 took effect. 100% bfloat16 -> it silently")
    print("     no-op'd and stage 2 will OOM here (that exact bug was hit and fixed once).")
elif vram_gb < 60:
    print("40GB-class GPU: use the reduced settings in Part 2 (num_frames=121, 480x832).")
else:
    print("Big GPU: you can use full README settings in Part 2 (num_frames=241, 736x1280).")


Thu Jul 23 07:55:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   25C    P0             46W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import os, sys, glob, subprocess, shutil
from pathlib import Path

# --- FIX 3: never let the kernel sit in a deleted directory ---
os.chdir("/content")

REPO_ROOT = "/content/JoyAI-Echo"
OUTPUT_DIR = f"{REPO_ROOT}/inference_result"

# Both checkpoints cache to LOCAL disk (/content, 256GB). Drive is a FUSE
# network mount -- reading the 46GB Echo checkpoint through it at load time
# was the actual bottleneck (GPU/RAM idle while it crawled through Drive I/O).
# Local disk costs a fresh download each new Colab runtime instead, but every
# load within the session is then fast local-disk I/O.
LOCAL_HF_CACHE = "/content/hf_cache_local"

os.makedirs(LOCAL_HF_CACHE, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

def run(cmd, cwd=None, label=""):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        print(f"--- FAILED: {label or ' '.join(cmd)} (exit {r.returncode}) ---")
        print(r.stdout[-2000:])
        print(r.stderr[-3000:])
        raise RuntimeError(f"Step failed: {label or cmd}")
    return r

# --- HF auth from Colab secret 'hf' ---
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("hf")
print("HF token loaded from Colab secret 'hf'.")

# --- FIX 4: clone only counts if requirements.txt is actually there ---
if not os.path.exists(f"{REPO_ROOT}/requirements.txt"):
    shutil.rmtree(REPO_ROOT, ignore_errors=True)
    print("Cloning repo...")
    run(["git", "clone", "https://github.com/vermasachin6102/JoyAI-Echo.git", REPO_ROOT], label="git clone")
else:
    print("Repo present and complete, skipping clone.")

for sub in ["ltx-core/src", "ltx-pipelines/src", "ltx-distillation/src"]:
    p = os.path.join(REPO_ROOT, sub)
    if p not in sys.path:
        sys.path.insert(0, p)

# --- System deps ---
print("Installing ffmpeg...")
run(["apt-get", "-qq", "update"], label="apt update")
run(["apt-get", "-qq", "install", "-y", "ffmpeg"], label="apt install ffmpeg")

# --- Python deps (pinned CUDA 12.8 stack) ---
print("Installing pinned torch stack (this takes a few minutes)...")
# --force-reinstall is required, not optional: Colab's base image ships a
# newer torch preinstalled, and `pip install torch==2.8.0` without it can
# exit 0 (success) while silently leaving the existing version in place --
# pip's resolver may decide the already-installed version satisfies some
# other preinstalled package's constraint and skip the actual downgrade.
# Confirmed on a genuinely fresh A100 runtime (not a stale-kernel issue).
run(["pip", "install", "--quiet", "--force-reinstall", "--index-url", "https://download.pytorch.org/whl/cu128",
     "torch==2.8.0", "torchvision==0.23.0", "torchaudio==2.8.0"], label="torch install")

# Fail loudly and immediately if the pin didn't stick (e.g. a stale kernel
# session still has an old torch cached from an earlier run in memory --
# `import torch` returns the cached module, not what's freshly on disk).
# Catching this here beats discovering it several steps later as a cryptic
# xformers ABI crash.
import torch as _torch_check
if not _torch_check.__version__.startswith("2.8.0"):
    raise RuntimeError(
        f"torch is {_torch_check.__version__}, expected 2.8.0 -- the pinned "
        "install didn't take effect in this process. Usually means a stale "
        "kernel: Runtime -> Restart session, then rerun all cells from the top."
    )
print(f"torch version confirmed: {_torch_check.__version__}")
del _torch_check
print("Installing requirements.txt...")
run(["pip", "install", "--quiet", "-r", "requirements.txt"], cwd=REPO_ROOT, label="requirements.txt")
# --- FIX 2: constrained hub install AFTER requirements, never unpinned -U ---
print("Installing huggingface_hub (constrained <1.0 for transformers 4.57.6)...")
run(["pip", "install", "--quiet", "huggingface_hub[cli]>=0.34.0,<1.0"], label="hf hub install")

import torch
print("torch:", torch.__version__, "| CUDA:", torch.version.cuda, "| available:", torch.cuda.is_available())

# --- Checkpoints ---
from huggingface_hub import snapshot_download
os.makedirs(f"{REPO_ROOT}/checkpoints", exist_ok=True)

# Echo checkpoint (~46GB) — LOCAL disk. Resumable if interrupted; re-download
# is the cost of avoiding slow Drive-FUSE reads during every model load.
print("Fetching JoyAI-Echo release checkpoint to local disk...")
echo_dir = snapshot_download(
    repo_id="jdopensource/JoyAI-Echo",
    cache_dir=LOCAL_HF_CACHE,
    allow_patterns=["*.safetensors", "*.json", "*.md"],
)
candidates = glob.glob(os.path.join(echo_dir, "**", "*.safetensors"), recursive=True)
assert candidates, "No .safetensors found in jdopensource/JoyAI-Echo"
dst_echo = f"{REPO_ROOT}/checkpoints/echo-longvideo-release.safetensors"
if os.path.islink(dst_echo) or os.path.exists(dst_echo):
    os.remove(dst_echo)
os.symlink(candidates[0], dst_echo)
print("Echo checkpoint ->", os.path.realpath(dst_echo))

# --- FIX 1: the repo's inference.yaml actually looks for checkpoints/test.safetensors ---
dst_test = f"{REPO_ROOT}/checkpoints/test.safetensors"
if os.path.islink(dst_test) or os.path.exists(dst_test):
    os.remove(dst_test)
os.symlink(dst_echo, dst_test)
print("Config-compat symlink: test.safetensors ->", os.path.realpath(dst_test))

# Gemma (~24GB) — LOCAL disk. Resumable if interrupted.
print("Fetching gemma-3-12b-it to local disk (gated — needs accepted license)...")
gemma_dir = snapshot_download(repo_id="google/gemma-3-12b-it", cache_dir=LOCAL_HF_CACHE)
dst_gemma = f"{REPO_ROOT}/checkpoints/gemma-3-12b"
if os.path.islink(dst_gemma):
    os.remove(dst_gemma)
elif os.path.isdir(dst_gemma):
    shutil.rmtree(dst_gemma)
os.symlink(gemma_dir, dst_gemma)
print("Gemma ->", os.path.realpath(dst_gemma))

# Gemma Q4_0 GGUF (~8GB) -- language-model-only quantized weights, used for the
# text encoder instead of the bf16 safetensors above. Loaded by requantizing to
# NF4 on GPU: ~13GB peak vs ~24GB for bf16, which OOMs on 24GB-class GPUs.
# The bf16 gemma dir above is still required -- the tokenizer/processor files
# come from there, not from the GGUF (it holds weights only).
print("Fetching gemma-3-12b Q4_0 GGUF (gated -- same license as above)...")
from huggingface_hub import hf_hub_download
GGUF_PATH = hf_hub_download(
    repo_id="google/gemma-3-12b-it-qat-q4_0-gguf",
    filename="gemma-3-12b-it-q4_0.gguf",
    cache_dir=LOCAL_HF_CACHE,
)
print("Gemma GGUF ->", GGUF_PATH)

# Verify every file the pipeline will open actually resolves
print("\n--- Verification ---")
ok = True
for label, path in [
    ("config's checkpoint (test.safetensors)", dst_test),
    ("gemma shard 5 (the one that kept failing)", f"{dst_gemma}/model-00005-of-00005.safetensors"),
    ("gemma tokenizer", f"{dst_gemma}/tokenizer.model"),
    ("gemma Q4_0 GGUF (text encoder)", GGUF_PATH),
]:
    exists = os.path.exists(path)  # follows symlinks — catches broken links
    print(("OK  " if exists else "MISSING  ") + label)
    ok = ok and exists
print("\nSETUP COMPLETE — go to Part 2." if ok else "\nSetup incomplete — re-run this cell (downloads resume).")

In [ ]:
# Fast-iteration cell: pulls latest code from GitHub only -- no deps
# reinstall, no checkpoint re-download. Re-run this alone after pushing
# code changes to main, then jump straight to Part 2.
print("Pulling latest code...")
r = run(["git", "pull", "origin", "main"], cwd=REPO_ROOT, label="git pull")
print(r.stdout.strip() or r.stderr.strip() or "(no output)")

for sub in ["ltx-core/src", "ltx-pipelines/src", "ltx-distillation/src"]:
    p = os.path.join(REPO_ROOT, sub)
    if p not in sys.path:
        sys.path.insert(0, p)


In [ ]:
import json, glob, subprocess, time
from pathlib import Path
from IPython.display import Video, display

def generate_video(
    prompts,
    name="my_story",
    seed=42,
    num_frames=121,
    video_height=480,
    video_width=832,
    preview=True,
    seed_video=None,   # path to a seed .mp4 (e.g. a Veo 3 clip copied to Drive) to prime the memory bank
    gemma_gguf=GGUF_PATH,  # NF4-quantized text encoder (~13GB vs ~24GB bf16). None = full bf16.
    fp8_generator=None,    # None = auto: on below 40GB VRAM (needed to fit), off above (bf16 is
                           # more precise and already fits). True/False to force either way.
):
    # fp8 costs precision, so only default it on where bf16 demonstrably won't fit.
    # vram_gb comes from the GPU-check cell at the top of the notebook.
    use_fp8 = (vram_gb < 40) if fp8_generator is None else fp8_generator
    # Write prompts to JSON, run inference.py, return path to the output .mp4.
    # Full stderr is printed on failure — no more hidden tracebacks.
    prompts_dir = Path(REPO_ROOT) / "prompts"
    prompts_dir.mkdir(parents=True, exist_ok=True)
    prompt_file = prompts_dir / f"{name}.json"
    with open(prompt_file, "w") as f:
        json.dump({"prompts": prompts}, f, indent=2)
    print(f"Wrote {len(prompts)} shot(s) to {prompt_file}")

    cmd = ["python", "inference.py",
           "--prompts-glob", prompt_file.name,
           "--num-frames", str(num_frames),
           "--video-height", str(video_height),
           "--video-width", str(video_width),
           "--seed", str(seed)]
    if seed_video:
        cmd += ["--seed-video", str(seed_video)]
    if gemma_gguf:
        cmd += ["--gemma-gguf-path", str(gemma_gguf)]
    cmd += ["--quantization-fp8-enabled", "true" if use_fp8 else "false"]
    print("Running:", " ".join(cmd))
    t0 = time.time()
    # Stream stdout live instead of buffering -- subprocess.run(capture_output=True)
    # hides everything until the process exits, so per-step/bottleneck logs never
    # show up during the run. Popen + line-by-line read prints them as they happen.
    proc = subprocess.Popen(
        cmd, cwd=REPO_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    lines = []
    for line in proc.stdout:
        print(line, end="")
        lines.append(line)
    returncode = proc.wait()
    print(f"inference.py exited {returncode} after {time.time() - t0:.0f}s")
    if returncode != 0:
        print("----- last 60 lines -----")
        print("".join(lines[-60:]))
        return None

    outputs = sorted(glob.glob(f"{OUTPUT_DIR}/outputs/**/*.mp4", recursive=True),
                     key=lambda p: Path(p).stat().st_mtime)
    if not outputs:
        print("No .mp4 produced — check the log above.")
        return None
    latest = outputs[-1]
    print("Output video:", latest)
    if preview:
        display(Video(latest, embed=True, width=640))
    return latest

print("generate_video() ready — go to Part 2.")

In [ ]:
prompts = [
    # Single short shot — speed test only (intro seed + one 10s extension)
    "PIERRE the sky-blue cartoon parrot lands on DINO the mint-green baby dinosaur's head. In a cheerful voice, Pierre says, \"Bonjour, Dino!\" Bright 2D cartoon style, sunny meadow background, cheerful children's music.",
]

# TODO: set this to your Veo 3 clip's path on Drive, e.g.:
#   "/content/drive/MyDrive/veo3_clips/intro.mp4"
# Leave as None to skip seeding and generate from prompts alone.
SEED_VIDEO_PATH = None  # <-- placeholder: paste your Veo 3 Drive video path here

video_path = generate_video(
    prompts,
    name="french_lesson",
    seed=42,
    num_frames=497,      # ~19.9s @ 25fps (nearest valid 1+8k frame count to 20s)
    video_height=736,
    video_width=1280,
    seed_video=SEED_VIDEO_PATH,
)

In [ ]:
import shutil, os
from google.colab import files

if video_path:
    if os.path.isdir("/content/drive/MyDrive"):
        keep_dir = "/content/drive/MyDrive/joyai-echo-outputs"
        os.makedirs(keep_dir, exist_ok=True)
        kept = shutil.copy(video_path, keep_dir)
        print("Copied to Drive:", kept)
    files.download(video_path)